# Stage 04 Decision System Demo

This notebook demonstrates the Stage 04 decision-system layer for `secfsn_plus_decisionsystems`.

Stages 01–03 happen upstream. They prepare the company-period panel and may optionally produce model risk scores.

Stage 04 consumes those inputs and produces decision-ready outputs.

The canonical handoff is:

```text
prepared panel -> merge_model_outputs(...) -> run_decision_engine(...)
```

This demo shows two paths:

1. Deterministic-only Stage 04, with no model risk probability.
2. Model-assisted Stage 04, where partial model outputs are merged into the prepared panel.

## 1. Load the upstream-shaped fixture

For the demo, we use a tiny synthetic fixture committed with the repo.

In the real pipeline, this prepared panel would come from Stages 01–03. For the video, the fixture lets us show the Stage 04 handoff clearly without rerunning upstream processing.

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Resolve repo root whether notebook is run from repo root, notebooks/, or Notebooks/
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    REPO_ROOT = CURRENT_DIR.parent
else:
    REPO_ROOT = CURRENT_DIR

# Add repo root to Python import path for notebook execution
repo_root_str = str(REPO_ROOT)
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

from secfsn_plus_decisionsystems.integration.model_bridge import merge_model_outputs
from secfsn_plus_decisionsystems.integration.decision_engine import run_decision_engine

print(f"Notebook cwd: {CURRENT_DIR}")
print(f"Repo root:    {REPO_ROOT}")

Notebook cwd: c:\Users\RAS\OneDrive\RAS\Trading\Live\secfsn_plus_decisionsystems\Notebooks
Repo root:    c:\Users\RAS\OneDrive\RAS\Trading\Live\secfsn_plus_decisionsystems


In [11]:
fixture_path = REPO_ROOT / "tests" / "fixtures" / "upstream_prepared_panel.csv"

print(fixture_path)
print("Exists:", fixture_path.exists())

c:\Users\RAS\OneDrive\RAS\Trading\Live\secfsn_plus_decisionsystems\tests\fixtures\upstream_prepared_panel.csv
Exists: True


In [3]:
raw_panel = pd.read_csv(fixture_path)

raw_panel

,cik,period,equity_ni_residual_scaled,cash_vs_earnings_gap_scaled,accrual_intensity,structural_drift_1q,margin_volatility_4q,balance_sheet_identity_residual_scaled,retained_earnings_gap_proxy_scaled
0,1001,2024Q1,0.06,0.08,0.09,0.05,0.07,0.04,0.05
1,1001,2024Q2,0.09,0.11,0.12,0.07,0.08,0.05,0.06
2,2002,2024Q1,0.48,0.52,0.58,0.50,0.47,0.44,0.41
3,2002,2024Q2,0.83,0.86,0.88,0.84,0.82,0.80,0.77


## 2. Convert to the canonical Stage 04 index

Stage 04 expects a prepared panel keyed by:

```text
(cik, period)
```

That means every row represents one company-period observation.

In [4]:
prepared_panel = raw_panel.set_index(["cik", "period"])

preview_columns = [
    "equity_ni_residual_scaled",
    "cash_vs_earnings_gap_scaled",
    "accrual_intensity",
]

prepared_panel[preview_columns]

equity_ni_residual_scaled  cash_vs_earnings_gap_scaled  \
cik  period                                                           
1001 2024Q1                       0.06                         0.08   
     2024Q2                       0.09                         0.11   
2002 2024Q1                       0.48                         0.52   
     2024Q2                       0.83                         0.86   

             accrual_intensity  
cik  period                     
1001 2024Q1               0.09  
     2024Q2               0.12  
2002 2024Q1               0.58  
     2024Q2               0.88

## 3. Run Stage 04 in deterministic-only mode

The model probability column is optional.

This first run shows that Stage 04 can produce decision outputs using only deterministic upstream features.

In [5]:
deterministic_decisions = run_decision_engine(prepared_panel)

base_decision_columns = [
    "structural_risk_score",
    "fraud_proxy_score",
    "hybrid_risk_score",
    "long_allowed",
    "short_allowed",
    "risk_override",
]

deterministic_decisions[base_decision_columns]

structural_risk_score  fraud_proxy_score  hybrid_risk_score  \
cik  period                                                                
1001 2024Q1               0.065000              0.070           0.067500   
     2024Q2               0.086667              0.090           0.088333   
2002 2024Q1               0.498333              0.495           0.496667   
     2024Q2               0.838333              0.825           0.831667   

             long_allowed  short_allowed  risk_override  
cik  period                                              
1001 2024Q1          True          False          False  
     2024Q2          True          False          False  
2002 2024Q1         False          False          False  
     2024Q2         False           True          False

## 4. Create optional partial model outputs

Now we simulate Stage 03 / Layer 3A model outputs.

Only two rows receive `model_risk_probability`. This shows partial model coverage: Stage 04 should keep every prepared-panel row, even when only some rows have model scores.

In [6]:
model_outputs = pd.DataFrame(
    {
        "cik": [1001, 2002],
        "period": ["2024Q2", "2024Q2"],
        "model_risk_probability": [0.18, 0.93],
    }
).set_index(["cik", "period"])

model_outputs

,,model_risk_probability
cik,period,
1001,2024Q2,0.18
2002,2024Q2,0.93


## 5. Merge optional model outputs into the prepared panel

The bridge keeps the canonical Stage 04 shape.

The output remains indexed by `(cik, period)`, and uncovered rows stay present with missing model probability values.

In [7]:
merged_panel = merge_model_outputs(prepared_panel, model_outputs)

merged_panel[preview_columns + ["model_risk_probability"]]

equity_ni_residual_scaled  cash_vs_earnings_gap_scaled  \
cik  period                                                           
1001 2024Q1                       0.06                         0.08   
     2024Q2                       0.09                         0.11   
2002 2024Q1                       0.48                         0.52   
     2024Q2                       0.83                         0.86   

             accrual_intensity  model_risk_probability  
cik  period                                             
1001 2024Q1               0.09                     NaN  
     2024Q2               0.12                    0.18  
2002 2024Q1               0.58                     NaN  
     2024Q2               0.88                    0.93

## 6. Run the decision engine with optional model outputs

Now we run the same Stage 04 decision engine after merging partial model scores.

The important point is that the final output remains keyed by the same company-period index and still produces canonical decision columns.

In [8]:
model_decision_columns = [
    "structural_risk_score",
    "fraud_proxy_score",
    "model_risk_probability",
    "hybrid_risk_score",
    "long_allowed",
    "short_allowed",
    "risk_override",
]

In [9]:
model_assisted_decisions = run_decision_engine(merged_panel)

model_assisted_decisions[model_decision_columns]

structural_risk_score  fraud_proxy_score  model_risk_probability  \
cik  period                                                                     
1001 2024Q1               0.065000              0.070                     NaN   
     2024Q2               0.086667              0.090                    0.18   
2002 2024Q1               0.498333              0.495                     NaN   
     2024Q2               0.838333              0.825                    0.93   

             hybrid_risk_score  long_allowed  short_allowed  risk_override  
cik  period                                                                 
1001 2024Q1           0.067500          True          False          False  
     2024Q2           0.088333          True          False          False  
2002 2024Q1           0.496667         False          False          False  
     2024Q2           0.831667         False           True          False

## 7. Compact final demo table

For the video, this is the main table to show.

It demonstrates that Stage 04 produces decision-ready outputs while preserving the original company-period key.

In [10]:
final_demo_table = model_assisted_decisions[model_decision_columns].copy()

final_demo_table

structural_risk_score  fraud_proxy_score  model_risk_probability  \
cik  period                                                                     
1001 2024Q1               0.065000              0.070                     NaN   
     2024Q2               0.086667              0.090                    0.18   
2002 2024Q1               0.498333              0.495                     NaN   
     2024Q2               0.838333              0.825                    0.93   

             hybrid_risk_score  long_allowed  short_allowed  risk_override  
cik  period                                                                 
1001 2024Q1           0.067500          True          False          False  
     2024Q2           0.088333          True          False          False  
2002 2024Q1           0.496667         False          False          False  
     2024Q2           0.831667         False           True          False

## What this demo proves

This Stage 04 repo is the decision-system layer.

It does not rerun Stages 01–03. Instead, it consumes their prepared outputs.

This walkthrough shows that Stage 04:

- accepts a prepared panel indexed by `(cik, period)`
- works without model probabilities
- can merge optional partial model outputs
- keeps all prepared-panel rows
- produces compact decision-ready outputs
- preserves the company-period key through the handoff